In [1]:
### Import des modules
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import utils
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [2]:
data = pd.read_csv('data/rafined/employees.csv', sep=',')

In [3]:
# Plus un employé a d'ancienneté, moins il va avoir tendance à partir
data['ratio_anciennete_total_experence'] = np.where(
    data['annee_experience_totale'] > 0,
    data['annees_dans_l_entreprise'] / data['annee_experience_totale'],
    data['annees_dans_l_entreprise']
)

# Plus un employé évolue au sein de l'entreprise, moins il va avoir tendance à partir
data['ratio_poste_actuel_anciennete'] = np.where(
    data['annees_dans_l_entreprise'] > 0,
    data['annees_dans_le_poste_actuel'] / data['annees_dans_l_entreprise'],
    data['annees_dans_le_poste_actuel']
)

# Indicateur de pénibilité de déplacement distance * frequence (1=jamais, 1.5=occasionnel, 2=frequent)
data['score_penibilite_transport'] = data['distance_domicile_travail'] * data['frequence_deplacement']

# Plus le score bien-être est élevée moins un salarié va avoir tendance a quitter l'entreprise
colonnes_satisfaction = [
    'satisfaction_employee_environnement',
    'satisfaction_employee_nature_travail',
    'satisfaction_employee_equipe',
    'satisfaction_employee_equilibre_pro_perso'
]
data['score_bien_etre'] = data[colonnes_satisfaction].mean(axis=1)

# Plus le score de peformance est élevée plus un salarié va rester
colonnes_evaluation = [
    'note_evaluation_precedente',
    'note_evaluation_actuelle'
]
data['score_performance'] = data[colonnes_evaluation].mean(axis=1)

# Si l'évalusation est à la baisse, baisse des résultats donc salarié moins satisfait ?
data['evolution_performance'] = data['note_evaluation_actuelle'] - data['note_evaluation_precedente']

#
data['evolution_hierarchie_score'] = np.where(
    data['annee_experience_totale'] > 0,
    data['niveau_hierarchique_poste'] / data['annee_experience_totale'],
    data['niveau_hierarchique_poste']
)

# Plus un employé s'en va rapidement de ses anciennes entreprises, plus il aura tendance à partir rapidement ?
data['annee_par_experience'] = np.where(
    data['nombre_experiences_precedentes'] > 0,
    data['annee_experience_totale'] / data['nombre_experiences_precedentes'],
    data['annee_experience_totale']
)

# Création d'une colonne pour savoir comment se positione le salarié en salaire par rapport à ses collègues au même poste
data['quartile_salaire_par_poste'] = data.groupby('poste')['revenu_mensuel'].transform(
    lambda x: pd.qcut(x, q=4, labels=[1, 2, 3, 4], duplicates='drop')
)

In [4]:
train_data = utils.split_train_data(data, 'a_quitte_l_entreprise')

In [5]:
preprocessor = utils.create_preprocessor()

In [6]:
dummy_model = DummyClassifier(strategy="most_frequent", random_state=42)
random_logistic_regression = RandomForestClassifier(random_state=42, class_weight='balanced')
random_forest_model = LogisticRegression(random_state=42, max_iter=1000)

models = [
    {'name' : 'DummyClassifier', 'model': dummy_model},
    {'name' : 'LogisticRegression', 'model': random_logistic_regression},
    {'name' : 'RandomForestClassifier', 'model': random_forest_model},
]

utils.benchmark(models, preprocessor, train_data)

Benchmarking model: DummyClassifier
Accuracy moyenne : 0.8316336098088712
Recall moyen : 0.0
F1-Score moyen : 0.0
------------------
Benchmarking model: LogisticRegression
Accuracy moyenne : 0.8520627479264334
Recall moyen : 0.37384615384615383
F1-Score moyen : 0.45742162907587175
------------------
Benchmarking model: RandomForestClassifier
Accuracy moyenne : 0.8724594302199785
Recall moyen : 0.4447435897435897
F1-Score moyen : 0.5370676085608171
------------------
